# Delta-filter analytical order and cutoff dependence

This notebook compares the analytical estimate `K ≈ 2.65499√(1-λ²)/δ` with the order resolved by the current production builder. The estimate is a cutoff-independent search bracket; it is not the final filter contract.

In [ ]:
import Pkg
const NOTEBOOK_ROOT = normpath(joinpath(@__DIR__, ".."))
Pkg.activate(NOTEBOOK_ROOT)

using UniPolfed
using LinearAlgebra, Random
const RUN_PLOTS = false


## Smoke-sized QREM

`L=8` keeps exact diagonalization quick. The physical eigenvalues are affinely mapped to `[-1, 1]`, which is the coordinate system used by the filters.

In [ ]:
"""Dense QREM: diag(ε) + (hx/2) Σᵢ σᵢˣ."""
function qrem_hamiltonian(L::Int; hx::Float64=1.0, seed::Integer=1234)
    dimension = 1 << L
    diagonal = sqrt(L / 2) .* randn(MersenneTwister(seed), dimension)
    H = zeros(Float64, dimension, dimension)
    @inbounds for state in 0:(dimension - 1)
        H[state + 1, state + 1] = diagonal[state + 1]
        for bit in 0:(L - 1)
            neighbour = xor(state, 1 << bit)
            state < neighbour && (H[state + 1, neighbour + 1] = hx / 2)
        end
    end
    return Symmetric(H, :U)
end

L = 8
H = qrem_hamiltonian(L)
eigenvalues = eigvals(H)
Emin, Emax = extrema(eigenvalues)
spread, center = (Emax - Emin) / 2, (Emax + Emin) / 2
mapped_eigenvalues = (eigenvalues .- center) ./ spread
println("L=$L, dimension=$(size(H, 1)), spectrum=[$Emin, $Emax]")


## Resolve orders over a cutoff sweep

A candidate is retained only when `build_filter` succeeds and `validate_filter` accepts its coverage/leakage contract. Unexpected errors remain visible.

In [ ]:
target = 0.0
half_width = 0.03
requested_interval = UniPolfed.Interval(target - half_width, target + half_width)
cutoffs = collect(0.17:0.03:0.32)

function resolved_order(target::Real, half_width::Real, cutoff::Real)
    half_width > 0 || throw(ArgumentError("half_width must be positive"))
    interval = UniPolfed.Interval(target - half_width, target + half_width)
    -1 <= interval.left < interval.right <= 1 || throw(ArgumentError("window must lie in [-1, 1]"))
    candidate = try
        UniPolfed.build_filter(UniPolfed.DeltaUpFilter, interval; cutoff)
    catch error_value
        occursin("passband exceeds", sprint(showerror, error_value)) && return missing
        rethrow()
    end
    return UniPolfed.validate_filter(candidate) ? UniPolfed.polynomial_order(candidate) : missing
end

analytical_seed = UniPolfed.DeltaUpFilter(requested_interval; cutoff=first(cutoffs))
analytical_K = UniPolfed.analytical_order(analytical_seed, half_width)
resolved_orders = [resolved_order(target, half_width, cutoff) for cutoff in cutoffs]
@assert any(!ismissing, resolved_orders)
collect(zip(cutoffs, resolved_orders))


In [ ]:
selected_index = findfirst(!ismissing, resolved_orders)
selected_cutoff = cutoffs[selected_index]
filter = UniPolfed.build_filter(
    UniPolfed.DeltaUpFilter, requested_interval; cutoff=selected_cutoff,
)
P = UniPolfed._scalar_filter(filter)
levels = P.(mapped_eigenvalues)
inside = (requested_interval.left .<= mapped_eigenvalues) .&
         (mapped_eigenvalues .<= requested_interval.right)

println("analytical bracket estimate: $(round(analytical_K; digits=2))")
println("resolved order at cutoff $selected_cutoff: $(filter.order)")
println("resolved passband: [$(filter.passband.left), $(filter.passband.right)]")
println("exact QREM states in requested interval: $(count(inside))")
@assert UniPolfed.validate_filter(filter)


## Optional plots

The first panel shows the selected filter over the rescaled QREM spectrum. The second shows how the validated resolved order changes with cutoff while the analytical bracket stays fixed.

In [ ]:
if RUN_PLOTS
    using PythonPlot
    xgrid = range(-1.0, 1.0; length=2_000)
    fig, axes = subplots(1, 2; figsize=(12, 4), constrained_layout=true)

    axes[0].plot(xgrid, P.(xgrid); label="P_K(x), K=$(filter.order)")
    axes[0].scatter(mapped_eigenvalues, levels; s=9, c=inside, cmap="coolwarm")
    axes[0].axhline(filter.cutoff; color="tab:red", linestyle="--", label="cutoff")
    axes[0].axvspan(requested_interval.left, requested_interval.right; color="tab:green", alpha=0.15)
    axes[0].set(xlabel="rescaled energy", ylabel="P_K(x)", title="selected filter")
    axes[0].legend(fontsize=8)

    valid = findall(!ismissing, resolved_orders)
    axes[1].plot(cutoffs[valid], Float64[resolved_orders[i] for i in valid], "o-"; label="resolved order")
    axes[1].axhline(analytical_K; color="black", linestyle="--", label="analytical bracket")
    axes[1].set(xlabel="cutoff", ylabel="order K", title="cutoff dependence")
    axes[1].legend(fontsize=8)
    fig
end


### Interpretation

The analytical expression seeds the bracket-and-bisect search. The production order additionally depends on the cutoff and safety factor, and the resolved passband can be asymmetric away from the center. Treat agreement in scale—not exact equality—as the expected result.